Libraries

In [1]:
#Importar librerias
import numpy as np
import pandas as pd
import os
import csv
import matplotlib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
import pyodbc
import openpyxl
import pickle
import itertools
from pandas import to_datetime
from datetime import timedelta, date, datetime
import math
pd.options.display.max_rows = 999

In [2]:
# from src.model import Model
# from src.simulation import simulate_patients
# from src.utils import get_labelled_sequences

# data = simulate_patients(
#     freq=5,      # sampling frequency of the time series, in minutes
#     length=84,   # length of the time series, in days
#     num=100,     # number of time series
# )

# # reshape the dataset from long to wide
# data = data.pivot(index='ts', columns=['id'], values=['gl'])
# data.columns = data.columns.get_level_values(level='id')

Importing Data

In [3]:
#carga y procesamiento de los datos
f = r'C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables/HDeviceCGM.txt'
# "C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/CGM_Editada3.txt"
list_to_append = []
for chunk in pd.read_csv(f,  sep='|', header=0, chunksize=100000):
    list_to_append.append(chunk)
MasterDF = pd.concat(list_to_append)
# MasterDF = pd.read_csv(f, sep='|', header=0, low_memory = False,chunksize=100000 )
# MasterDF = MasterDF[MasterDF.PtID.isin([183, 184,  14, 220, 233,  62,  17, 186,  52, 216, 115,  37, 244,])]
MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']

Feature Engineering

In [4]:
#Creating a date time column
MasterDF['Today'] = datetime.today().date()
MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
#selecting just the columns for Giammarino's code to run
MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
MasterDF= MasterDF.reset_index(drop=True)
MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
MasterDF['Dia_Noche'] = MasterDF['ts'].dt.hour.between(7, 18, inclusive='both') \
    .replace({True: 'Dia', False: 'Noche'})
    # MasterDF = MasterDF[MasterDF['Dia_Noche']=='Dia']
MasterDF=MasterDF[['ts', 'id','gl']]
data = MasterDF

C:\Users\Anderson Mosquera\AppData\Local\Temp\ipykernel_46360\892241908.py:3: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')


In [5]:
data = MasterDF#[MasterDF['id'].isin([183])]

In [6]:
data

,ts,id,gl
0,2024-10-15 05:35:00,183,162.0
1,2024-10-15 05:30:00,183,164.0
2,2024-10-15 05:25:00,183,168.0
3,2024-10-15 05:20:00,183,169.0
4,2024-10-15 05:15:00,183,170.0
...,...,...,...
14807430,2025-02-03 08:47:00,293,210.0
14807431,2025-02-03 08:42:00,293,211.0
14807432,2025-02-03 08:37:00,293,210.0
14807433,2025-02-03 08:32:00,293,207.0


In [7]:
# data = data[data['gl'].between(40, 400)]
# # reshape the dataset from long to wide
# data = data.pivot(index='ts', columns=['id'], values=['gl'])
# data.columns = data.columns.get_level_values(level='id')
# data.reset_index(inplace = True)
# data['date'] = pd.to_datetime(data['ts']).dt.date
# # data

In [8]:
# from src.New_Utils import New_Sequences

In [9]:
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
from pandas import to_datetime
from datetime import timedelta, date, datetime
import math
import pandas as pd


def Event(Data,glucose_threshold):
    'Data has to be type list'
    'Threshold should be an integer'
    C1 = 0
    C2 = 0
    for i in range(len(Data)) :
        if Data[i] < glucose_threshold:
            C1+=1
        if C1 == 3:
            C2+=1
        elif C1 >2:
            pass
        else:
            pass
    else:
        C1=0
    if C2 > 1:
        return 1
    else:
        return 0

# create a list for storing the data

def New_Sequences(data):
    sequences = []
    NotWorking = []
    minutes = 5
    Days_Week = 7 # this is the number of days to be considered i na week, can be changed
    glucose_threshold = 54
    # calculate the number of timestamps in one week
    sequence_length = int(Days_Week * 24 * 60 // minutes)
    for patient in data.id.unique():
        # Data = pd.DataFrame()
        if type(patient) != 0:
            
            try:
                #Do preprocessing inside the function per patient
                Data = data[data['id'].isin([patient])]
                Data = data[data['gl'].between(40, 400)]
                # reshape the dataset from long to wide
                Data = Data.pivot(index='ts', columns=['id'], values=['gl'])
                Data.columns = Data.columns.get_level_values(level='id')
                Data.reset_index(inplace = True)
                Data['date'] = pd.to_datetime(Data['ts']).dt.date
                Data=Data[Data[patient].notnull()]
                #Per Patient
                min_Date = datetime.strptime(str(Data['ts'].min())[:-9], '%Y-%m-%d').date()
                # print(min_Date)
                max_Date = datetime.strptime(str(Data['ts'].max())[:-9], '%Y-%m-%d').date()
                # print(max_Date)
                Difference = abs(max_Date-min_Date).days #difference in days between the two dates  
                Sequences = math.ceil(Difference/Days_Week) # Define the number of 1 week sequences
                Result = pd.DataFrame()
                for i in range (1, Sequences, 1):
                    # generate the range
                    date_generated = pd.DataFrame()    
                    date_generated = [min_Date + timedelta(days=x) for x in range(0, (timedelta(days=Days_Week)).days)]
                    # print(len(date_generated))
                    min_Date = min_Date + timedelta(days=Days_Week+1)
                    df_Generated = pd.DataFrame(date_generated)
                    df_Generated = df_Generated.rename(columns={0:'date'})
                    df_Generated = pd.merge(df_Generated,Data,'left',left_on='date',right_on='date')
                    df_Generated['Sequence'] = i
                    #Aqui le voy a meter la interpolacion lineal
                    Result = pd.concat([Result,df_Generated])
                Grouped = Result.groupby('Sequence').count()
                Grouped['Total_Readings_Sequence'] = sequence_length
                Grouped = Grouped[['ts','Total_Readings_Sequence']]
                Grouped['Time_Worn'] = Grouped['ts']/Grouped['Total_Readings_Sequence']
                Grouped['Criteria'] = np.where(Grouped['Time_Worn']>0.7, "Applicable", 'Not Applicable')
                Grouped = Grouped[Grouped['Criteria']=='Applicable']
                Result = Result[Result['Sequence'].isin(Grouped.index.to_series())]          
                for i in Grouped.index.to_series():
                    #Evaluate if next is applicable
                    try:
                        if Grouped.loc[i+1]['Criteria'] == 'Applicable':
                            X = Result[patient][Result['Sequence'] == i].dropna().to_list()
                            L = len(X)
                            Y = Result[patient][Result['Sequence'] == i+1].to_list()
                            Y = Event(Y, glucose_threshold)
                            #I need to determine the amount of consecutive data below the threshold and if it greater that 15 minutes (three readings) then Y = 1
                            # save the patient's data
                            sequences.append({
                            'patient': patient,
                            'Sequence': i,
                            'start': str(Result['ts'][Result['Sequence'] == i].min()),
                            'end': str(Result['ts'][Result['Sequence'] == i].max()),
                            'L': L,
                            'X': X,
                            'Y': Y
                            })
                        else:
                            pass
                    except:
                        pass
            except:
                NotWorking.append(patient)
        else:
            pass
    return sequences

In [10]:
sequences = New_Sequences(data)

In [38]:
#Balancing the Dataset
import random
positives = 0
sequences_bal = []
sequences_neg = []
for i in range(len(sequences)):
    if sequences[i]['Y'] == 1:
        positives += 1
        sequences_bal.append(sequences[i])
    else:
        sequences_neg.append(sequences[i])
positives
random.shuffle(sequences_neg)
len(sequences_neg[:positives])
sequences_bal.extend(sequences_neg[:positives])
len(sequences_bal)
sequences = sequences_bal

In [18]:
import itertools
l1_penalty=[0.001, 0.005, 0.01]
l2_penalty=[0.01, 0.05, 0.1]
batch_size = [16, 32, 64]

Parameters = []
# crear una matriz con las combinaciones de parametros
for element in itertools.product(l1_penalty,l2_penalty,batch_size):
    Parameters.append(list(element))
len(Parameters)

27

In [19]:
def Run_Models(sequences, Parameters, skf, Model):
    l1_penalty=Parameters[0]
    l2_penalty=Parameters[1]
    batch_size=Parameters[2]
    
    results = []

    # loop across the folds
    for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
        
        # fit the model to the training set
        model = Model()

        model.fit(
            sequences=[sequences[i] for i in train_index],
            sequence_length=int(7 * 24 * 60 // 5),
            l1_penalty=l1_penalty,
            l2_penalty=l2_penalty,
            learning_rate=0.00001, #leave still
            batch_size=batch_size,
            epochs=1000,
            seed=42,
            verbose=0
        )

        # evaluate the model on the test set
        metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

        # save the results
        results.append(metrics)

    # organize the results in a data frame
    results = pd.DataFrame(results)

    return [Parameters, results.mean()]

In [20]:
# to run the experiments
from src.model import Model
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
Global_Results = []
for i in [x for x in range (len(Parameters))]:
    try:
        R = Run_Models(sequences, Parameters[i], skf, Model)
        Global_Results.append(
            {
                'Parameters: ': R[0],
                'Results: ': R[1]
            }
        )
    except:
        continue

In [21]:
#Extracting the results
Global_Results

[{'Parameters: ': [0.001, 0.01, 16],
  'Results: ': accuracy             0.617122
  balanced_accuracy    0.520224
  precision            0.193788
  sensitivity          0.370386
  specificity          0.670061
  f1                   0.254281
  auc                  0.535734
  dtype: float64},
 {'Parameters: ': [0.001, 0.01, 32],
  'Results: ': accuracy             0.616284
  balanced_accuracy    0.510016
  precision            0.185187
  sensitivity          0.345714
  specificity          0.674318
  f1                   0.241047
  auc                  0.531234
  dtype: float64},
 {'Parameters: ': [0.001, 0.01, 64],
  'Results: ': accuracy             0.612106
  balanced_accuracy    0.507834
  precision            0.183205
  sensitivity          0.346631
  specificity          0.669038
  f1                   0.239613
  auc                  0.529860
  dtype: float64},
 {'Parameters: ': [0.001, 0.05, 16],
  'Results: ': accuracy             0.614445
  balanced_accuracy    0.525257
  preci

In [37]:
k=0
for i in Global_Results:
    print(str(Global_Results[k]['Parameters: ']) +'-'+str(Global_Results[k]['Results: '].values[6]))
    k+=1

[0.001, 0.01, 16]-0.535734369997265
[0.001, 0.01, 32]-0.5312344879237381
[0.001, 0.01, 64]-0.529860446154119
[0.001, 0.05, 16]-0.5468716757058442
[0.001, 0.05, 32]-0.5410404580925631
[0.001, 0.05, 64]-0.5381958652035961
[0.001, 0.1, 16]-0.5531752430727754
[0.001, 0.1, 32]-0.5479300024594316
[0.001, 0.1, 64]-0.5446533117269632
[0.005, 0.01, 16]-0.5647058342355619
[0.005, 0.01, 32]-0.5621275072543895
[0.005, 0.01, 64]-0.5617532440003629
[0.005, 0.05, 16]-0.5691893849594363
[0.005, 0.05, 32]-0.5663596611185218
[0.005, 0.05, 64]-0.5659604754512688
[0.005, 0.1, 16]-0.5725540284825548
[0.005, 0.1, 32]-0.5693885794633958
[0.005, 0.1, 64]-0.5686117480459344
[0.01, 0.01, 16]-0.574182272032859
[0.01, 0.01, 32]-0.5731817215833761
[0.01, 0.01, 64]-0.5714985624349554
[0.01, 0.05, 16]-0.5744604729398305
[0.01, 0.05, 32]-0.5732551779856301
[0.01, 0.05, 64]-0.5714275455727144
[0.01, 0.1, 16]-0.5744591009277699
[0.01, 0.1, 32]-0.573609377181534
[0.01, 0.1, 64]-0.5712087815750577


In [15]:
from src.model import Model
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.01,
        l2_penalty=0.1,
        learning_rate=0.00001, #leave still
        batch_size=16,
        epochs=418,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.558762
balanced_accuracy    0.547626
precision            0.207379
sensitivity          0.530395
specificity          0.564858
f1                   0.297844
auc                  0.571817
dtype: float64


experiments

In [12]:
from src.model import Model
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.01,
        l2_penalty=0.1,
        learning_rate=0.01, #leave still
        batch_size=16,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.706227
balanced_accuracy    0.514091
precision            0.189074
sensitivity          0.217026
specificity          0.811157
f1                   0.196060
auc                  0.532606
dtype: float64


In [13]:
from src.model import Model
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.01,
        l2_penalty=0.1,
        learning_rate=0.1, #leave still
        batch_size=16,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.785816
balanced_accuracy    0.505136
precision            0.232182
sensitivity          0.071177
specificity          0.939094
f1                   0.095327
auc                  0.540314
dtype: float64


In [14]:
from src.model import Model
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.01,
        l2_penalty=0.1,
        learning_rate=0.01, #leave still
        batch_size=16,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.706227
balanced_accuracy    0.514091
precision            0.189074
sensitivity          0.217026
specificity          0.811157
f1                   0.196060
auc                  0.532606
dtype: float64


In [40]:
#out of the experiments this is the best model so far
from src.model import Model
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.01,
        l2_penalty=0.1,
        learning_rate=0.00001, #leave still
        batch_size=16,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.526019
balanced_accuracy    0.526096
precision            0.527343
sensitivity          0.513226
specificity          0.538967
f1                   0.518990
auc                  0.550950
dtype: float64


In [42]:
#Defining Hyper Parameters
l1_penalty=[0.001, 0.01, 0.1]
l2_penalty=[0.001, 0.01, 0.1]
learning_rate = [0.000001, 0.0001, 0.1]
batch_size = [16, 32, 64]

Parameters = []
# crear una matriz con las combinaciones de parametros
for element in itertools.product(l1_penalty,l2_penalty,learning_rate,batch_size):
    Parameters.append(list(element))
len(Parameters)

81

In [54]:
Global_Results_2 = pd.DataFrame(columns=['Parameters','AUC'])

Global_Results_2 = Global_Results_2._append(
            {'Parameters': str([0.001,0.1,12,76]) , 'AUC': 0.7865 }, 
            ignore_index=True)
Global_Results_2 = Global_Results_2._append(
            {'Parameters': str([0.002,0.1,12,76]) , 'AUC': 0.7965 }, 
            ignore_index=True)
Global_Results_2 = Global_Results_2._append(
            {'Parameters': str([0.007,0.1,12,76]) , 'AUC': 0.5965 }, 
            ignore_index=True)
Global_Results_2.sort_values(by='AUC', ascending=False)

C:\Users\Anderson Mosquera\AppData\Local\Temp\ipykernel_46360\769097170.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Global_Results_2 = Global_Results_2._append(


,Parameters,AUC
1,"[0.002, 0.1, 12, 76]",0.7965
0,"[0.001, 0.1, 12, 76]",0.7865
2,"[0.007, 0.1, 12, 76]",0.5965
